# Reproduksi SOTANER

## Konfigurasi

In [ ]:
import os, sys, time, json, glob, platform, importlib, warnings, itertools
from pathlib import Path
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np

NB_DIR = Path.cwd()
PROJECT_ROOT   = NB_DIR.parent
CONLL2012_V4   = PROJECT_ROOT / "conll-2012" / "v4" / "data"

DATA_DIR = NB_DIR / "data"
BIO_DIR = DATA_DIR / "bio"
SPLITS_DIR = DATA_DIR / "bio-splits"
CROSSGENRE_DIR = BIO_DIR / "crossgenre"
PERTURB_DIR = BIO_DIR / "perturb"
SPARK_FMT_DIR = DATA_DIR / "spark-format"
SPACY_FMT_DIR = DATA_DIR / "spacy-fmt"
STANZA_FMT_DIR = DATA_DIR / "stanza-fmt"
RESULTS_DIR = NB_DIR / "results"
MODELS_DIR = NB_DIR / "models"
TMP_DIR = NB_DIR / "tmp"
for d in (DATA_DIR, RESULTS_DIR, MODELS_DIR, TMP_DIR):
    d.mkdir(exist_ok=True)
for sub in ("A", "B", "C", "D"):
    (RESULTS_DIR / sub).mkdir(exist_ok=True)

sys.path.insert(0, str(NB_DIR))
import helpers
from helpers import paper_values as PV
from helpers.report_utils import comparison_table, export_all, f1_of, summarise_folds

FULL_RUN = False   # Jika True maka di section 3 dijalankan penuh 10 fold, jika False hanya dijalankan 3 fold
RUN_SPARKNLP = True    # jalankan kolom Spark NLP
SEED = 42

N_FOLDS = 10 if FULL_RUN else 3
SPACY_MAX_STEPS = 20000 if FULL_RUN else 2000
STANZA_MAX_STEPS = 5000  if FULL_RUN else 2000
SPARK_MAX_EPOCHS = 10 if FULL_RUN else 1 
SPACY_GPU = os.environ.get("SOTANER_SPACY_GPU", "0") == "1"

GENRES, NEWS, ALL6 = helpers.GENRES, helpers.NEWS, helpers.ALL6
CROSS = helpers.CROSS_GENRES
ENTITY_TYPES = helpers.ENTITY_TYPES
LIBS = ["spacy", "stanza"] + (["sparknlp"] if RUN_SPARKNLP else [])

ALL_TABLES = {}

FULL_RUN = False | RUN_SPARKNLP = True | N_FOLDS = 3 | SEED = 42
conll-2012/v4 : d:\OneDrive\Penelitian\NER Methods Comparison\conll-2012\v4\data -> OK


In [5]:
import spacy, stanza, seqeval, faker, sklearn, scipy, pandas, numpy, matplotlib, openpyxl, pyspark, sparknlp

# Section 1 - Persiapan

### 1.1 Load dan Ubah Dataset menjadi Format BIO

In [ ]:
t0 = time.time()
report = helpers.build_all(str(CONLL2012_V4), str(BIO_DIR))
print(f"\nselesai dalam {time.time() - t0:.0f} detik -> {BIO_DIR}")

train        bc   sents= 10429  entities=  8654
train        bn   sents=  9723  entities= 17573
train        mz   sents=  6911  entities= 10921
train        nw   sents= 15288  entities= 35771
train        pt   sents= 15263  entities=     0
train        tc   sents= 11162  entities=  2233
train        wb   sents=  6411  entities=  6676
train        news sents= 31922  entities= 64265
train        all6 sents= 59924  entities= 81828

development  bc   sents=  1946  entities=  1459
development  bn   sents=  1172  entities=  2172
development  mz   sents=   642  entities=  1232
development  nw   sents=  2054  entities=  4883
development  pt   sents=  1075  entities=     0
development  tc   sents=  1634  entities=   311
development  wb   sents=  1080  entities=  1009
development  news sents=  3868  entities=  8287
development  all6 sents=  8528  entities= 11066

test         bc   sents=  2037  entities=  1697
test         bn   sents=  1252  entities=  2184
test         mz   sents=   780  entiti

### 1.2 Validasi jumlah kalimat dan entitas (mencocokan dengan paper)

In [5]:
from helpers.bio_utils import read_bio_file, count_entities, entity_type_counts

rows = []
for split in ["train", "development", "test"]:
    for name in ["news", "all6", *GENRES]:
        p = BIO_DIR / split / f"onto.{name}.ner"
        if p.exists():
            s, t = read_bio_file(str(p))
            rows.append({"split": split, "subset": name,
                         "sentences": len(s), "entities": count_entities(t)})
counts = pd.DataFrame(rows)
display(counts.pivot_table(index="subset", columns="split",
                           values=["sentences", "entities"], sort=False))

s, t = read_bio_file(str(BIO_DIR / "test" / "onto.all6.ner"))
assert len(s) == 8262 and count_entities(t) == 11257, (len(s), count_entities(t))
print(f"OK  test/onto.all6.ner = {len(s)} kalimat / {count_entities(t)} entitas "
      f"== support micro Tabel 3 paper (11.257)")
print("distribusi tipe entitas (test all6):")
display(pd.Series(dict(sorted(entity_type_counts(t).items(), key=lambda x: -x[1]))))

sentences                     entities                     
split      train development    test    train development     test
subset                                                            
news     31922.0      3868.0  3930.0  64265.0      8287.0   8043.0
all6     59924.0      8528.0  8262.0  81828.0     11066.0  11257.0
bc       10429.0      1946.0  2037.0   8654.0      1459.0   1697.0
bn        9723.0      1172.0  1252.0  17573.0      2172.0   2184.0
mz        6911.0       642.0   780.0  10921.0      1232.0   1163.0
nw       15288.0      2054.0  1898.0  35771.0      4883.0   4696.0
pt       15263.0      1075.0  1217.0      0.0         0.0      0.0
tc       11162.0      1634.0  1366.0   2233.0       311.0    380.0
wb        6411.0      1080.0   929.0   6676.0      1009.0   1137.0

OK  test/onto.all6.ner = 8262 kalimat / 11257 entitas == support micro Tabel 3 paper (11.257)
distribusi tipe entitas (test all6):


GPE            2240
PERSON         1988
ORG            1795
DATE           1602
CARDINAL        935
NORP            841
PERCENT         349
MONEY           314
TIME            212
ORDINAL         195
LOC             179
WORK_OF_ART     166
FAC             135
QUANTITY        105
PRODUCT          76
EVENT            63
LAW              40
LANGUAGE         22
dtype: int64

### 1.3 Tabel 2: *Performance of OntoNotes NER models in the three NLP libraries*

In [6]:
from helpers.eval_blackbox import load_spacy, load_stanza, evaluate_bio, run_blackbox

NLP = load_spacy()
STZ = load_stanza()

STD_TEST = str(BIO_DIR / "test" / "onto.all6.ner")
(RESULTS_DIR / "A").mkdir(exist_ok=True)

t0 = time.time()
std_report = evaluate_bio(STD_TEST, nlp=NLP, stanza_tagger=STZ)   # {'spacy': {...}, 'stanza': {...}}
print(f"eval spaCy + Stanza pada test standar: {time.time() - t0:.0f} dtk")

with open(RESULTS_DIR / "A" / "onto.all6.txt", "w", encoding="utf-8") as fh:
    for m in ("spacy", "stanza"):
        fh.write(f"Classification report for {m.capitalize()} NER:\n{std_report[m]['text']}\n\n")

obt_t2 = {"spacy": f1_of(std_report["spacy"]), "stanza": f1_of(std_report["stanza"])}
print({k: round(v, 2) for k, v in obt_t2.items()})

spaCy: CPU
spaCy model: en_core_web_trf
Stanza NER pipeline loaded (use_gpu=True)
eval spaCy + Stanza pada test standar: 1125 dtk
{'spacy': np.float64(89.19), 'stanza': np.float64(88.24)}


In [7]:
SPARK = SN_PIPELINE = sn_std_report = None
if RUN_SPARKNLP:
    from helpers import eval_sparknlp
    from helpers.spark_session import start_spark
    SPARK = start_spark(memory="8g")
    print("Spark", SPARK.version)
    SN_PIPELINE = eval_sparknlp.build_pipeline()          # bert_base_cased + onto_bert_base_cased
    sn_std = eval_sparknlp.evaluate_bio_sparknlp(
        [STD_TEST], spark=SPARK, pipeline=SN_PIPELINE,
        tmp_dir=str(SPARK_FMT_DIR), results_dir=str(RESULTS_DIR), tag="A")
    sn_std_report = sn_std["onto.all6"]
    obt_t2["sparknlp"] = f1_of(sn_std_report)
    print("Spark NLP micro-F1:", round(obt_t2["sparknlp"], 2))

spark env: JAVA_HOME=C:\Program Files\Java\jdk-17 (ok)  HADOOP_HOME=C:\hadoop (hadoop.dll ok)
Spark: offline jar spark-nlp-assembly-5.5.3.jar
Spark 3.5.9
BertEmbeddings.load(file:///C:/Users/hanif/cache_pretrained/bert_base_cased)
NerDLModel.load(file:///C:/Users/hanif/cache_pretrained/onto_bert_base_cased)
onto.all6 micro-F1=88.59  (dropped 0 rows)
Spark NLP micro-F1: 88.59


In [8]:
t2 = pd.DataFrame({
    "Reported on Paper": [PV.TABLE2_OBTAINED[l] for l in LIBS],
    "Obtained": [round(obt_t2[l], 2) for l in LIBS],
    "Delta":[round(obt_t2[l] - PV.TABLE2_OBTAINED[l], 2) for l in LIBS],
    "Reported (situs library)": [PV.TABLE2_WEBSITE[l] for l in LIBS],
}, index=[PV.LIB_LABEL[l] for l in LIBS])
t2.index.name = "Library"
ALL_TABLES["Tabel2_library_check"] = t2
display(t2)

,Reported on Paper,Obtained,Delta,Reported (situs library)
Library,,,,
spaCy,89.09,89.19,0.10,90.00
Stanza,88.71,88.24,-0.47,88.80
Spark NLP,88.60,88.59,-0.01,89.97


# SECTION 2 - Black-box Experiments

Evaluasi **model off-the-shelf** (tanpa pelatihan ulang) pada split test set standar OntoNote.
Semua metrik = F1 micro entity-level (seqeval). Model spaCy + Stanza dari Section 1 dipakai ulang.

### 2.1 Tabel 3: F-score per tipe entitas

In [ ]:
def type_f1(rep, et):
    return f1_of(rep, label=et)

obt_t3 = {}
for et in ENTITY_TYPES:
    obt_t3[et] = {"spacy": type_f1(std_report["spacy"], et),
                  "stanza": type_f1(std_report["stanza"], et)}
    if sn_std_report is not None:
        obt_t3[et]["sparknlp"] = type_f1(sn_std_report, et)

t3 = comparison_table("Entity type", ENTITY_TYPES, obt_t3, PV.TABLE3_ALL18, libs=LIBS)
ALL_TABLES["Tabel3_per_type"] = t3
display(t3)

,in paper Table 3,spaCy Reported,spaCy Obtained,spaCy Delta,Stanza Reported,Stanza Obtained,Stanza Delta,Spark NLP Reported,Spark NLP Obtained,Spark NLP Delta
Entity type,,,,,,,,,,
CARDINAL,,82.29,82.95,0.66,85.87,85.24,-0.63,85.54,85.54,-0.00
DATE,yes,85.63,86.87,1.24,86.55,85.49,-1.06,85.54,85.48,-0.06
EVENT,yes,74.42,70.31,-4.11,64.96,58.41,-6.55,53.23,52.03,-1.20
FAC,,74.71,75.64,0.93,73.56,70.23,-3.33,74.42,74.13,-0.29
GPE,yes,95.36,95.64,0.28,95.20,95.46,0.26,95.61,95.59,-0.02
LANGUAGE,yes,74.42,66.67,-7.75,60.61,60.61,-0.00,60.61,60.61,-0.00
LAW,yes,67.50,64.00,-3.50,64.79,58.82,-5.97,64.71,64.71,-0.00
LOC,,75.94,76.22,0.28,75.48,76.42,0.94,79.21,79.21,0.00
MONEY,,89.28,82.80,-6.48,89.03,89.03,0.00,87.07,87.07,-0.00


Catatan: tipe langka (LANGUAGE 22, LAW 40, EVENT 63, PRODUCT 76 gold) statistiknya berisik — paper pun menandai ini.


### 2.2 Tabel 4: performa per *source*

In [10]:
SRC = ["bn", "mz", "nw", "bc", "tc", "wb"]
src_files = [str(BIO_DIR / "test" / f"onto.{g}.ner") for g in SRC]

src_res, NLP, STZ = run_blackbox(src_files, str(RESULTS_DIR), "A", nlp=NLP, stanza_tagger=STZ)
sn_src = {}
if RUN_SPARKNLP:
    sn_src = eval_sparknlp.evaluate_bio_sparknlp(
        src_files, spark=SPARK, pipeline=SN_PIPELINE,
        tmp_dir=str(SPARK_FMT_DIR), results_dir=str(RESULTS_DIR), tag="A")

obt_t4 = {}
for g in SRC:
    stem = f"onto.{g}"
    obt_t4[g] = {"spacy": f1_of(src_res[stem]["spacy"]), "stanza": f1_of(src_res[stem]["stanza"])}
    if RUN_SPARKNLP:
        obt_t4[g]["sparknlp"] = f1_of(sn_src[stem])

t4 = comparison_table("Source", SRC, obt_t4, PV.TABLE4_SOURCE, libs=LIBS)
ALL_TABLES["Tabel4_per_source"] = t4
display(t4)

onto.bn spacy=91.95 stanza=91.92
onto.mz spacy=86.69 stanza=85.24
onto.nw spacy=90.98 stanza=90.84
onto.bc spacy=88.77 stanza=85.74
onto.tc spacy=76.78 stanza=75.03
onto.wb spacy=83.71 stanza=81.44
onto.bn micro-F1=90.93  (dropped 0 rows)
onto.mz micro-F1=87.73  (dropped 0 rows)
onto.nw micro-F1=90.94  (dropped 0 rows)
onto.bc micro-F1=87.59  (dropped 0 rows)
onto.tc micro-F1=78.11  (dropped 0 rows)
onto.wb micro-F1=80.11  (dropped 0 rows)


,spaCy Reported,spaCy Obtained,spaCy Delta,Stanza Reported,Stanza Obtained,Stanza Delta,Spark NLP Reported,Spark NLP Obtained,Spark NLP Delta
Source,,,,,,,,,
bn,91.64,91.95,0.31,91.82,91.92,0.10,90.93,90.93,-0.00
mz,88.72,86.69,-2.03,85.97,85.24,-0.73,87.73,87.73,0.00
nw,86.14,90.98,4.84,90.87,90.84,-0.03,90.96,90.94,-0.02
bc,91.55,88.77,-2.78,88.35,85.74,-2.61,87.59,87.59,0.00
tc,71.16,76.78,5.62,76.68,75.03,-1.65,78.38,78.11,-0.27
wb,82.81,83.71,0.90,81.20,81.44,0.24,80.11,80.11,-0.00


### 2.3 Tabel 5: performa per *genre*

Regrouping paper: **News** = `bn + mz + nw`; `bc`, `tc`, `wb`

In [11]:
news_file = str(BIO_DIR / "test" / "onto.news.ner")
news_res, NLP, STZ = run_blackbox([news_file], str(RESULTS_DIR), "A", nlp=NLP, stanza_tagger=STZ)
sn_news = {}
if RUN_SPARKNLP:
    sn_news = eval_sparknlp.evaluate_bio_sparknlp(
        [news_file], spark=SPARK, pipeline=SN_PIPELINE,
        tmp_dir=str(SPARK_FMT_DIR), results_dir=str(RESULTS_DIR), tag="A")

obt_t5 = {"news": {"spacy": f1_of(news_res["onto.news"]["spacy"]),
                   "stanza": f1_of(news_res["onto.news"]["stanza"])}}
if RUN_SPARKNLP:
    obt_t5["news"]["sparknlp"] = f1_of(sn_news["onto.news"])
for g in ["bc", "tc", "wb"]:
    obt_t5[g] = obt_t4[g]

t5 = comparison_table("Genre", ["news", "bc", "tc", "wb"], obt_t5, PV.TABLE5_GENRE, libs=LIBS)
ALL_TABLES["Tabel5_per_genre"] = t5
display(t5)

onto.news spacy=90.63 stanza=90.31
onto.news micro-F1=90.47  (dropped 0 rows)


,spaCy Reported,spaCy Obtained,spaCy Delta,Stanza Reported,Stanza Obtained,Stanza Delta,Spark NLP Reported,Spark NLP Obtained,Spark NLP Delta
Genre,,,,,,,,,
news,90.79,90.63,-0.16,90.41,90.31,-0.10,90.47,90.47,-0.00
bc,88.72,88.77,0.05,88.35,85.74,-2.61,87.59,87.59,0.00
tc,71.16,76.78,5.62,76.68,75.03,-1.65,78.37,78.11,-0.26
wb,82.81,83.71,0.90,81.20,81.44,0.24,80.11,80.11,-0.00


### 2.4 Tabel 6: *adversarial test sets*

Enam pertubation (konteks kalimat tak diubah, hanya token entitas ditulis ulang), ber-*seed*:

| ID | Transformasi |
|---|---|
| P1 | PERSON -> kata literal **"Dodo"** (tanpa Faker; uji memorisasi) |
| P2 | PERSON -> nama Faker **en_US** |
| P3 | PERSON -> nama Faker **en_IN** |
| P4 | PERSON -> nama **perempuan** Faker **en_TH** |
| P5 | PERSON -> nama **perempuan** Faker **en_IN** |
| P6 | GPE -> nama tempat Faker **en_IE** |


In [12]:
from helpers.perturb import make_all_perturbations

pert = make_all_perturbations(STD_TEST, str(PERTURB_DIR), seed=SEED)
pert_files = [pert[f"perturb{i}"] for i in range(1, 7)]

pr_res, NLP, STZ = run_blackbox(pert_files, str(RESULTS_DIR), "B", nlp=NLP, stanza_tagger=STZ)
sn_pert = {}
if RUN_SPARKNLP:
    sn_pert = eval_sparknlp.evaluate_bio_sparknlp(
        pert_files, spark=SPARK, pipeline=SN_PIPELINE,
        tmp_dir=str(SPARK_FMT_DIR), results_dir=str(RESULTS_DIR), tag="B")


def pf1(stem, lib, label):
    if lib == "sparknlp":
        return f1_of(sn_pert[stem], label=label)
    return f1_of(pr_res[stem][lib], label=label)


def base_f1(lib, label):
    rep = sn_std_report if lib == "sparknlp" else std_report[lib]
    return f1_of(rep, label=label)

rows_all, rows_per = {}, {}
rows_all["None"] = {l: base_f1(l, "micro avg") for l in LIBS}
rows_per["None"] = {l: base_f1(l, "PERSON") for l in LIBS}
for i in range(1, 6):
    stem = f"onto.all.test.perturb{i}"
    rows_all[f"P{i}"] = {l: pf1(stem, l, "micro avg") for l in LIBS}
    rows_per[f"P{i}"] = {l: pf1(stem, l, "PERSON") for l in LIBS}
rows_all["P6"] = {l: pf1("onto.all.test.perturb6", l, "micro avg") for l in LIBS}

rows_gpe = {"None": {l: base_f1(l, "GPE") for l in LIBS},
            "P6":   {l: pf1("onto.all.test.perturb6", l, "GPE") for l in LIBS}}

order_all = ["None", "P1", "P2", "P3", "P4", "P5", "P6"]
t6_all = comparison_table("Setting", order_all, rows_all, PV.TABLE6_ALL, libs=LIBS)
t6_all.insert(0, "perturbation", ["(baseline)"] + [PV.PERTURB_DEFS[f"P{i}"] for i in range(1, 7)])

t6_per = comparison_table("Setting (F1 kelas PERSON)", ["None", "P1", "P2", "P3", "P4", "P5"],
                          rows_per, PV.TABLE6_CLASS, libs=LIBS)
paper_gpe = {"None": PV.TABLE6_NONE_GPE, "P6": PV.TABLE6_CLASS["P6"]}
t6_gpe = comparison_table("Setting (F1 kelas GPE)", ["None", "P6"], rows_gpe, paper_gpe, libs=LIBS)

ALL_TABLES["Tabel6_All"] = t6_all
ALL_TABLES["Tabel6_PER"] = t6_per
ALL_TABLES["Tabel6_GPE"] = t6_gpe
display(t6_all); display(t6_per); display(t6_gpe)
print("Temuan paper yang diharapkan muncul: P1 menjatuhkan F1 PERSON (~93 -> ~83) = model menghafal token; "
      "P4 turun ~10 poin PERSON; P6 menjatuhkan F1 GPE (~95 -> ~65); P2/P3/P5 hanya ~1 poin di All.")

perturb1: PERSON literal:Dodo   locale=-      -> 3400 tokens rewritten  (d:\OneDrive\Penelitian\NER Methods Comparison\SOTANER Notebook Based\data\bio\perturb\onto.all.test.perturb1.ner)
perturb2: PERSON name           locale=en_US  -> 3400 tokens rewritten  (d:\OneDrive\Penelitian\NER Methods Comparison\SOTANER Notebook Based\data\bio\perturb\onto.all.test.perturb2.ner)
perturb3: PERSON name           locale=en_IN  -> 3400 tokens rewritten  (d:\OneDrive\Penelitian\NER Methods Comparison\SOTANER Notebook Based\data\bio\perturb\onto.all.test.perturb3.ner)
perturb4: PERSON name_female    locale=en_TH  -> 3400 tokens rewritten  (d:\OneDrive\Penelitian\NER Methods Comparison\SOTANER Notebook Based\data\bio\perturb\onto.all.test.perturb4.ner)
perturb5: PERSON name_female    locale=en_IN  -> 3400 tokens rewritten  (d:\OneDrive\Penelitian\NER Methods Comparison\SOTANER Notebook Based\data\bio\perturb\onto.all.test.perturb5.ner)
perturb6: GPE    gpe            locale=en_IE  -> 2868 tokens rewr

KeyboardInterrupt: 